# TeDA on Kaggle (2x T4)
Run the cells in order. Settings: **Accelerator = GPU T4 x2, Internet = On**. Every long step is resumable:
if the session stops, start it again and re-run the same cells (finished work is skipped).

In [ ]:
REPO_URL = "https://github.com/<your-user>/teda.git"   # <- edit
DATA_DIR = "/kaggle/working/vimd16k"                   # wav16k/ + manifest.tsv (see section 2)

## 1. Code

In [ ]:
%cd /kaggle/working
!git clone $REPO_URL teda 2>/dev/null || git -C teda pull
%cd /kaggle/working/teda
!pip install -q -e .
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Data (once)
**Option A**: attach your private Kaggle Dataset that contains `vimd16k/` (wav16k + manifest.tsv) and set
`DATA_DIR = "/kaggle/input/<dataset>/vimd16k"` in the first cell. **Option B**: build it here (about 12 GB output;
ViMD is CC-BY-NC-ND-4.0, so keep the resulting Dataset private).

In [ ]:
import os
if not os.path.exists(f"{DATA_DIR}/manifest.tsv"):
    !python scripts/prepare_data.py --out-dir $DATA_DIR --workers 4
    !python scripts/scan_text.py --manifest-raw $DATA_DIR/manifest_raw.tsv --out-dir outputs/scan --manifest-out $DATA_DIR/manifest.tsv
else:
    print("manifest found:", f"{DATA_DIR}/manifest.tsv")

Read `outputs/scan/unresolved.tsv`, `review_sample.tsv` and `stats.json` once before training.

In [ ]:
if not os.path.exists("artifacts/tokenizer/bpe1024.model"):
    !python scripts/build_tokenizer.py --data-dir $DATA_DIR      # commit artifacts/tokenizer/ to the repo afterwards
else:
    print("tokenizer found")

## 3. Checks before the first training run
`check_encoder.py` verifies that the real ViP-VL checkpoint loads into the encoder (look at the tensor coverage).
`probe_memory.py` measures peak GPU memory on ONE GPU; pick the largest `max_frames_per_batch` that leaves a few GiB free.

In [ ]:
!python scripts/check_encoder.py
!python scripts/probe_memory.py --frames 12000 18000 24000 32000

## 4. Baseline E00 (CTC only)

In [ ]:
FRAMES = 24000   # <- value chosen with probe_memory.py
!torchrun --nproc_per_node=2 scripts/train.py --exp e00_baseline --set data.data_dir=$DATA_DIR train.max_frames_per_batch=$FRAMES

## 5. Freeze search (which ViP-VL layers to freeze)
Edit `configs/sweep/freeze_search.yaml`, then run. Choose by **valid** WER only, then fix that setting for E01 to E04.
`--set` overrides (here the data path and batch size) are applied to every run.

In [ ]:
!python scripts/sweep.py --set data.data_dir=$DATA_DIR train.max_frames_per_batch=$FRAMES
!cat outputs/sweep_freeze/summary.tsv

## 6. Evaluate (valid). The test split needs `--final` and is for the final report only.

In [ ]:
!python scripts/evaluate.py --exp e00_baseline --split valid --set data.data_dir=$DATA_DIR